# 09 — Intra/Inter-Cluster Similarity (v2)

Measures how **cohesive** each cluster is and how **separated** clusters are
from each other, for all 6 unsupervised methods:
minilm_kmeans, roberta_kmeans, minilm_agglomerative, roberta_agglomerative,
minilm_dec, roberta_dec.

Cluster assignments are loaded from `results/cluster_labels_{name}.npy`
(saved by notebook 02) — no re-running of clustering algorithms here.
Cosine similarity is always computed in the **original embedding space**
(pre-UMAP, pre-DEC encoder) so that distances are comparable across methods.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

from utils import config
from utils.embeddings import load_cached

train_clean = pd.read_parquet(config.PROCESSED_DIR / "train_clean.parquet")
true_labels = train_clean["label"].to_numpy()

METHODS = ["minilm", "roberta"]
CLUSTERERS = ["kmeans", "agglomerative", "dec"]
METHOD_NAMES = [f"{m}_{c}" for m in METHODS for c in CLUSTERERS]

suffix = "full"
embeddings_by_method = {}
for method in METHODS:
    arr = load_cached(f"{method}_train_{suffix}_summary")
    assert arr is not None, f"Missing cached embeddings for '{method}'"
    embeddings_by_method[method] = arr

print(f"Methods: {METHOD_NAMES}")
print(f"Rows: {len(train_clean)}")
for m, e in embeddings_by_method.items():
    print(f"  {m}: shape {e.shape}")

In [ ]:
def cluster_similarity_stats(embeddings: np.ndarray, cluster_labels: np.ndarray):
    """
    Compute intra-cluster cohesion and inter-cluster centroid similarity.

    Args:
        embeddings: (N, D) float32 — original embedding space (pre-UMAP)
        cluster_labels: (N,) int — cluster assignments; -1 = noise (excluded)

    Returns:
        per_cluster_df: DataFrame with columns [cluster, size, mean_intra_sim,
                        min_intra_sim, max_intra_sim]
        inter_mat: (K, K) float32 centroid-to-centroid cosine similarity matrix
        cluster_ids: sorted list of non-noise cluster ids
        mean_inter_sim: mean off-diagonal inter-cluster similarity (nan if K < 2)
    """
    unique_clusters = sorted(c for c in np.unique(cluster_labels) if c != -1)
    if not unique_clusters:
        return pd.DataFrame(), np.array([[]]), [], float("nan")

    rows = []
    centroids = []
    for cid in unique_clusters:
        mask = cluster_labels == cid
        members = embeddings[mask].astype(np.float32)
        centroid = members.mean(axis=0, keepdims=True)
        centroids.append(centroid[0])
        sims = cosine_similarity(members, centroid).ravel()
        rows.append({
            "cluster": cid,
            "size": int(mask.sum()),
            "mean_intra_sim": float(sims.mean()),
            "min_intra_sim": float(sims.min()),
            "max_intra_sim": float(sims.max()),
        })

    per_cluster_df = pd.DataFrame(rows)
    cent_matrix = np.stack(centroids).astype(np.float32)
    inter_mat = cosine_similarity(cent_matrix)

    n = len(unique_clusters)
    mean_inter = float(inter_mat[~np.eye(n, dtype=bool)].mean()) if n > 1 else float("nan")

    return per_cluster_df, inter_mat, unique_clusters, mean_inter

print("cluster_similarity_stats() defined.")

In [ ]:
all_stats = {}       # method_name -> (per_cluster_df, inter_mat, cluster_ids)
method_summary = []  # one row per method for the cross-method table

# Load majority-label lookup from clusters_{name}.csv (saved by notebook 02)
cluster_name_map = {}
for name in METHOD_NAMES:
    path = config.RESULTS_DIR / f"clusters_{name}.csv"
    if path.exists():
        cdf = pd.read_csv(path)
        cluster_name_map[name] = dict(zip(cdf["cluster"], cdf["majority_true_label"]))

for name in METHOD_NAMES:
    embedding_key = name.split("_")[0]  # "minilm" or "roberta"
    emb = embeddings_by_method[embedding_key]

    label_path = config.RESULTS_DIR / f"cluster_labels_{name}.npy"
    assert label_path.exists(), (
        f"Missing {label_path.name} — run notebook 02 first to generate cluster labels"
    )
    cluster_labels = np.load(label_path).astype(int)

    per_df, inter_mat, cids, mean_inter = cluster_similarity_stats(emb, cluster_labels)

    if per_df.empty:
        print(f"{name}: no non-noise clusters — skipping")
        continue

    mean_intra = float(per_df["mean_intra_sim"].mean())
    n_noise = int((cluster_labels == -1).sum())
    coverage = 1.0 - n_noise / len(cluster_labels)
    ratio = mean_intra / mean_inter if mean_inter and not np.isnan(mean_inter) else float("nan")

    all_stats[name] = (per_df, inter_mat, cids)
    method_summary.append({
        "method": name,
        "n_clusters": len(cids),
        "coverage": coverage,
        "mean_intra_sim": mean_intra,
        "mean_inter_sim": mean_inter,
        "separation_ratio": ratio,
    })
    print(f"{name}: intra={mean_intra:.3f}  inter={mean_inter:.3f}  "
          f"ratio={ratio:.3f}  coverage={coverage:.2f}  ({len(cids)} clusters)")

In [ ]:
for method_name, (cdf, _, _) in all_stats.items():
    cname_map = cluster_name_map.get(method_name, {})
    display = cdf.copy()
    display.insert(1, "majority_label",
                   display["cluster"].map(cname_map).fillna("?"))
    print(f"\n=== {method_name} ===")
    print(display.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

In [ ]:
summary_df = pd.DataFrame(method_summary).sort_values("separation_ratio", ascending=False)
print("=== Cross-method similarity comparison ===")
print(summary_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(config.RESULTS_DIR / "cluster_similarity_summary.csv", index=False)
print(f"\nSaved: results/cluster_similarity_summary.csv")

# Per-method centroid heatmaps
for method_name, (cdf, inter_mat, cids) in all_stats.items():
    if len(cids) < 2:
        print(f"{method_name}: too few clusters for heatmap — skipping")
        continue

    cname_map = cluster_name_map.get(method_name, {})
    tick_labels = [cname_map.get(c, str(c)) for c in cids]
    n = len(cids)

    fig, ax = plt.subplots(figsize=(max(6, n * 0.6), max(5, n * 0.55)))
    im = ax.imshow(inter_mat, vmin=0, vmax=1, cmap="YlOrRd", aspect="auto")
    plt.colorbar(im, ax=ax, label="Cosine similarity")
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=7)
    ax.set_yticklabels(tick_labels, fontsize=7)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f"{inter_mat[i, j]:.2f}",
                    ha="center", va="center",
                    fontsize=6 if n > 10 else 7,
                    color="white" if inter_mat[i, j] > 0.6 else "black")
    ax.set_title(f"{method_name} — cluster centroid cosine similarity", pad=10)
    plt.tight_layout()
    out = config.RESULTS_DIR / f"cluster_sim_heatmap_{method_name}.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved: {out.name}")

In [ ]:
plot_df = summary_df[summary_df["mean_intra_sim"].notna()].set_index("method")
x = np.arange(len(plot_df))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - width / 2, plot_df["mean_intra_sim"], width,
       label="Intra-cluster (\u2191 tight)", color="steelblue")
ax.bar(x + width / 2, plot_df["mean_inter_sim"], width,
       label="Inter-cluster centroid (\u2193 separated)", color="tomato")
ax.set_xticks(x)
ax.set_xticklabels(plot_df.index, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("Mean cosine similarity")
ax.set_title("Intra- vs. inter-cluster similarity by method")
ax.set_ylim(0, 1.05)
ax.legend()

for i, (_, row) in enumerate(plot_df.iterrows()):
    if not np.isnan(row["separation_ratio"]):
        ax.text(i, max(row["mean_intra_sim"], row["mean_inter_sim"]) + 0.02,
                f"\u00d7{row['separation_ratio']:.2f}",
                ha="center", fontsize=7, color="#333")

plt.tight_layout()
out = config.RESULTS_DIR / "cluster_similarity_comparison.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {out.name}")